<a href="https://colab.research.google.com/github/gabrielhierro/LinguagensDeProgramacao/blob/main/AtividadePraticaFolium/Atividade_Pratica_Folium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática: Visualização Geoespacial com Folium
### Contexto do Problema
Você está desenvolvendo um módulo de inteligência geográfica para mapear oportunidades imobiliárias e analisar a distribuição de propriedades nos municípios de Nova Iguaçu e Queimados. Utilizando uma base de dados de captação de imóveis, seu objetivo é criar um mapa interativo que permita à equipe de vendas visualizar a localização exata das propriedades, identificar concentrações de ofertas através de clusters e analisar o valor de mercado de forma espacial.



##1. Configuração do Ambiente e Base de Dados:


In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


## Parte 1: Inicialização e Marcadores Básicos

1. Crie um mapa base **(folium.Map)** centralizado na coordenada média de todos os imóveis do DataFrame. Ajuste o nível de zoom inicial **(zoom_start)** para 12 e utilize o estilo de mapa padrão **(OpenStreetMap)**.


In [7]:
m = folium.Map(location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()], zoom_start=12, tiles='OpenStreetMap')
m


2. Itere sobre as 5 primeiras linhas do DataFrame e adicione um marcador simples **(folium.Marker)** para cada imóvel. Configure o popup para exibir o tipo do imóvel e o valor de venda.



In [8]:
m = folium.Map(location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()], zoom_start=12, tiles='OpenStreetMap')

for index, row in df_mapa.head().iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Tipo: {row['tipo']}, Valor: R$ {row['valor_venda']}"
    ).add_to(m)

m

## Parte 2: Customização Visual com Marcadores Circulares
3. Crie um novo mapa base. Desta vez, utilize folium.CircleMarker para representar todos os imóveis da base. 4. Configure os marcadores circulares com as seguintes regras de negócio:
  *   **Raio:** Fixo em 8 pixels.
  *   **Cor de preenchimento:** Azul para 'Nova Iguaçu' e Laranja para 'Queimados'.
  *   **Tooltip:** Exiba o texto **"Clique para detalhes"** ao passar o mouse.




In [12]:
m2 = folium.Map(location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()], zoom_start=12, tiles='OpenStreetMap')

for index, row in df_mapa.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color='black',
        fill=True,
        fill_color='blue' if row['cidade'] == 'Nova Iguaçu' else 'orange',
        tooltip='Clique para detalhes'
    ).add_to(m2)

m2

## Parte 3: Agrupamento Inteligente (Clustering)
5. Quando há muitos pontos próximos, o mapa pode ficar poluído. Crie um terceiro mapa e instancie um objeto **MarkerCluster()**.


In [15]:
m3 = folium.Map(location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()], zoom_start=12, tiles='OpenStreetMap')
marker_cluster = MarkerCluster().add_to(m3)

for index, row in df_mapa.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color='black',
        fill=True,
        fill_color='blue' if row['cidade'] == 'Nova Iguaçu' else 'orange',
        tooltip='Clique para detalhes'
    ).add_to(marker_cluster)

m3

6. Adicione todos os imóveis do DataFrame a esse cluster **(em vez de adicioná-los diretamente ao mapa base)**. Utilize ícones customizados **(folium.Icon)** variando a cor conforme o tipo de imóvel **(ex: verde para Casa, azul para Apartamento, cinza para Terreno)**.


In [16]:
m4 = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Criando o cluster
marker_cluster = MarkerCluster().add_to(m4)

# Definindo a cor do ícone conforme o tipo do imóvel
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

# Adicionando todos os imóveis ao cluster
for index, row in df_mapa.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"""
        <b>ID:</b> {row['id_imovel']}<br>
        <b>Cidade:</b> {row['cidade']}<br>
        <b>Tipo:</b> {row['tipo']}<br>
        <b>Valor:</b> R$ {row['valor_venda']:,.2f}
        """,
        tooltip=f"{row['tipo']} - {row['cidade']}",
        icon=folium.Icon(
            color=cores_tipo[row['tipo']],
            icon='home' if row['tipo'] == 'Casa'
                  else 'building' if row['tipo'] == 'Apartamento'
                  else 'map-marker'
        )
    ).add_to(marker_cluster)

m4

7. Salve o mapa final em um arquivo HTML chamado **mapa_imoveis_baixada.html**.

In [17]:
# Salvando o mapa final em um arquivo HTML
m4.save('mapa_imoveis_baixada.html')

print('Arquivo mapa_imoveis_baixada.html gerado!')

m4


Arquivo mapa_imoveis_baixada.html gerado!
